In [ ]:
import enrichment as enr

Enrich the selected train set with different types of enrichment strategies:
Random, Toxic, Irony , and both.

In [ ]:
import pandas as pd
import enrichment as enr
import importlib
importlib.reload(enr)
train_path = "C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/LLM/datasets/Riloff/train_data.csv"

source_probs = "C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/BERTmodels/scores_results_finetuned/aux_labels/Ptacek/test_data_filtered.csv"
source_path = "C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/LLM/datasets/Ptacek/test_data_filtered.csv"

for enrichment_size in [0.5,1]:
    for i in range(5):
        j = int(enrichment_size*100)
        enrichment = enr.enrichment_randomly(train_path,source_path,enrichment_size,balance= False)
        enrichment.to_csv(f"C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/LLM/datasets/Riloff/upsampling_train_imbalanced/Riloff{j}_random{i}.csv", index=False, header=True)

        enrichment = enr.enrichment_single_label(train_path,source_path,source_probs,enrichment_size,"toxicity_probs",False,balance= False)
        enrichment.to_csv(f"C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/LLM/datasets/Riloff/upsampling_train_imbalanced/Riloff{j}_txc_neg{i}.csv", index=False, header=True)

        enrichment = enr.enrichment_single_label(train_path,source_path,source_probs,enrichment_size,"irony_probs",False,balance= False)
        enrichment.to_csv(f"C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/LLM/datasets/Riloff/upsampling_train_imbalanced/Riloff{j}_irony_neg{i}.csv", index=False, header=True)

        enrichment = enr.enrichment_two_label(train_path,source_path,source_probs,enrichment_size,"irony_probs","toxicity_probs",False,balance= False)
        enrichment.to_csv(f"C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/LLM/datasets/Riloff/upsampling_train_imbalanced/Riloff{j}_irony_txc_neg{i}.csv", index=False, header=True)


Train the enriched datasets on the selected testing sets.

In [ ]:
folder = "Riloff/upsampling_train_imbalanced/Riloff100_"
test = "SemEval2022/test_data_filtered.csv SARC/test_data_filtered.csv MUStARD/test_data_filtered.csv SPIRS/test_data_filtered.csv iSarcasmEval/test_data_filtered.csv multimodal_sarcasm_detection/test_data_filtered.csv" 

data_type = ["irony_neg","txc_neg","irony_txc_neg","random"]
nums = ['0','1','2','3','4']
for t in data_type:
    for num in nums:
        train_path = folder + t + num + ".csv"

        dataset = '"' + train_path + ' ' + test + '"'
        # print(dataset)
        !python runBERTmodels.py  --datasets {dataset}
    

Save the results by averaging the 5 iterations:

In [ ]:
enr.save_results(["random","toxicity","irony","toxicity and irony"],['100'],["Riloff100_random","Riloff100_txc_neg","Riloff100_irony_neg","Riloff100_irony_txc_neg"], "/scores_results_upsampling_neg/Riloff/upsampling_imbalanced_train_imbalanced/",5)

Plot the F1 score results of the enrichment :

In [ ]:
import pandas as pd

table = pd.read_csv("C:/Users/chanm/Desktop/studies/masters/Thesis/Thesis/BERTmodels/scores_results_upsampling_neg/Riloff/upsampling_train_imbalanced/mean.csv")

thresholds = [0.06, 0.101, 0.026, 0.098, 0.076, 0.073] #riloff

f1_imbal = pd.DataFrame(table.pivot(index='downsample type', columns='test data', values='mean_f1'))

f1_imbal.reset_index(inplace=True, drop=True)
enr.plot_f1_downsample(table,thresholds,"after Imbalanced Negative UpSampling Riloff Train Set from Ptacek Test Set","f1")
